# 22e — the (mu, gamma) -> distribution map: interactive grid explorer

**Question (Anuar).** Understand, concretely, how the Monte-Carlo **distribution** depends on `mu` and
`gamma` — and how that dependence changes with the **estimator** and where the **real data** sits.

**What this notebook builds.**
- A **grid** of `(mu, gamma)` values; for every node we generate the **2-D `(FWHM, sigma_fit)` cloud**
  with N_MC runs (same frozen noise draws for all estimators, so the comparison is fair).
- **Three estimators**, selectable: `ours: Lorentzian MLE` (the current pipeline), `ours: pseudo-Voigt`
  (the 21b experiment), and `Gregor: binned Voigt LSQ` (the experiment's own estimator, verbatim).
- **Static views** (always render): heatmaps of the grid-level summary statistics, small-multiples of
  the clouds, and an estimator-overlay comparison at representative nodes.
- **An interactive explorer** (ipywidgets): two knobs (**mu** and **gamma**) with `- / +` buttons and
  sliders (absolute values, no ratios), plus a **real-data** dropdown (`off` or any of the 14
  experiments) and a **fitter** dropdown.

**Fixed (the knobs are only mu and gamma):** the count-noise condition `sigma_prop`, `lambda` are pinned
to a labelled representative setting (`1nW Trans60`). `mu` and `gamma` are the absolute values.

Figures render **inline only** (no savefig); the notebook is executed **in place**.


## Panel
- **FIG 1** grid heatmaps (per estimator): median FWHM, median sigma_fit, IQR(FWHM) over the (mu, gamma) grid.
- **FIG 2** small-multiples: the 2-D cloud at a coarse (mu, gamma) sub-grid.
- **FIG 3** estimator overlay at representative nodes (ours-Lorentzian vs ours-pseudo-Voigt vs Gregor-Voigt).
- **FIG 4 (interactive)** the explorer: mu / gamma knobs + real-data dropdown + fitter dropdown.


In [1]:
# ============================================================
# 22e — imports
# ============================================================
import math, time, os, sys
import numpy as np
import torch

torch.set_default_dtype(torch.float32)
for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p); REPO_ROOT = _p; break
os.chdir(REPO_ROOT)

import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE
import matplotlib.pyplot as plt

from src.fitting import fit_profile, fwhm_from_theta, nll
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma
from src import fitting_lmfit as FL

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter

pio.templates.default = 'plotly_white'
pio.renderers.default = os.environ.get('PLOTLY_RENDERER', 'vscode')

print('Imports OK | plotly', plotly.__version__, '| lmfit', FL.lmfit.__version__)


Imports OK | plotly 7.1.0 | lmfit 1.3.4


In [2]:
# ============================================================
# EXPERIMENTS — true values from Gregor's fits (identical to 22a/22b/22c) + real data files
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]
print(len(EXPERIMENTS), 'experiments')


14 experiments


In [3]:
# ============================================================
# 22e CONFIG — the (mu, gamma) grid, the pinned noise condition, the estimators
# ============================================================
GRID_MU    = [3, 4, 6, 8, 12, 17, 24, 34, 48, 68, 96, 136, 192]      # photons (absolute)
GRID_GAMMA = [3, 4, 5, 6.5, 8.5, 11, 14, 18, 24, 31]                 # MHz   (absolute)

# The candidate-generation noise is NOT a knob: pinned to a representative mid/high-T condition.
COND_SIGMA_PROP = 9.851      # 1nW Trans60
COND_LAMBDA     = 2.593
COND_NAME       = '1nW Trans60  (sigma_prop=9.851, lambda=2.593)'

N_MC = 2000                  # Monte-Carlo runs per grid node (same frozen draws for all estimators)
SEED = 42

BIN_WIDTH  = 2.2             # MHz/step (Gregor's estimator)
WINDOW     = 75.0
MIN_COUNTS = 3

ESTIMATORS = [
    ('ours: Lorentzian MLE',      'lorentzian'),
    ('ours: pseudo-Voigt fit',    'pseudo_voigt'),
    ('Gregor: binned Voigt LSQ',  'gregor'),
]

SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    N_MC = 60
    GRID_MU    = GRID_MU[::3]
    GRID_GAMMA = GRID_GAMMA[::3]
    print('*** SMOKE RUN ***')

N_WORKERS = 4
# static sub-sampling for the small-multiples / comparison figures (index-based -> SMOKE-safe)
SUB_IDX_MU    = sorted(set([0, len(GRID_MU) // 3, 2 * len(GRID_MU) // 3, len(GRID_MU) - 1]))
SUB_IDX_GAMMA = sorted(set([0, len(GRID_GAMMA) // 2, len(GRID_GAMMA) - 1]))
def _idx(L, frac): return min(len(L) - 1, max(0, int(round(frac * (len(L) - 1)))))
NODES = [(_idx(GRID_MU, 0.25), _idx(GRID_GAMMA, 0.44)),
         (_idx(GRID_MU, 0.50), _idx(GRID_GAMMA, 0.44)),
         (_idx(GRID_MU, 0.50), _idx(GRID_GAMMA, 0.66))]
print(f'config: {len(GRID_MU)}x{len(GRID_GAMMA)} grid, N_MC={N_MC}, workers={N_WORKERS}')
print(f'  mu grid    : {GRID_MU}')
print(f'  gamma grid : {GRID_GAMMA}')
print(f'  pinned condition: {COND_NAME}')


config: 13x10 grid, N_MC=2000, workers=4
  mu grid    : [3, 4, 6, 8, 12, 17, 24, 34, 48, 68, 96, 136, 192]
  gamma grid : [3, 4, 5, 6.5, 8.5, 11, 14, 18, 24, 31]
  pinned condition: 1nW Trans60  (sigma_prop=9.851, lambda=2.593)


In [4]:
# ============================================================
# Estimators + one-node Monte-Carlo
#   our Lorentzian MLE      : unbinned Lorentzian, implicit diff  (the current pipeline)
#   our pseudo-Voigt fit    : 21b's fit model (fixed eta=0.2, 3 params)
#   Gregor binned Voigt LSQ : lmfit Voigt+constant on a 2.2 MHz-binned spectrum (verbatim 22c)
# ============================================================
def _fit_lor(ph):  return fit_profile(ph, n_iters=80, model='lorentzian',   uniform_bg=False)
def _fwhm_lor(th): return fwhm_from_theta(th, model='lorentzian')
def _nll_lor(th, ph): return nll(th, ph, model='lorentzian', uniform_bg=False)

def _fit_pv(ph):   return fit_profile(ph, n_iters=80, model='pseudo-voigt', uniform_bg=False)
def _fwhm_pv(th):  return fwhm_from_theta(th, model='pseudo-voigt')
def _nll_pv(th, ph): return nll(th, ph, model='pseudo-voigt', uniform_bg=False)

def lmfit_extract(photons_np, bin_width=BIN_WIDTH, window=WINDOW):
    """Gregor's estimator (verbatim from 21d/22c): bin, least-squares Voigt+constant, return (FWHM, stderr)."""
    lo, hi = -window / 2.0, window / 2.0
    edges = np.arange(lo, hi + bin_width, bin_width)
    counts, edges = np.histogram(photons_np, bins=edges)
    if counts.size == 0 or counts.max() < MIN_COUNTS:
        return None, None
    x = 0.5 * (edges[:-1] + edges[1:])
    m = FL.voigt
    p = m.make_params()
    p['amplitude'].set(value=max(float(counts.sum()), 1.0), min=0.0)
    p['center'].set(value=float(x[int(np.argmax(counts))]), min=lo, max=hi)
    p['sigma'].set(value=3.0, min=0.05, max=50.0)
    p['gamma'].set(value=3.0, min=0.05, max=100.0)
    p['c'].set(value=float(np.median(counts)), min=0.0)
    try:
        out = m.fit(counts, p, x=x)
    except Exception:
        return None, None
    fw = out.params['fwhm'].value; se = out.params['fwhm'].stderr
    fw = float(fw) if fw is not None and np.isfinite(fw) else None
    se = float(se) if se is not None and np.isfinite(se) else None
    return fw, se

def _mc_lorentzian(args):
    gamma, u, b = args
    return compute_fwhm_and_dgamma(gamma, u, b, _fit_lor, _fwhm_lor, _nll_lor, n_params=2)

def _mc_pseudo_voigt(args):
    gamma, u, b = args
    return compute_fwhm_and_dgamma(gamma, u, b, _fit_pv, _fwhm_pv, _nll_pv, n_params=3)

def _mc_gregor(args):
    gamma, u, b = args
    ph = build_photons(torch.tensor(float(gamma)), torch.as_tensor(u, dtype=torch.float32),
                       torch.as_tensor(b, dtype=torch.float32)).numpy()
    fw, se = lmfit_extract(ph)
    return (np.nan if fw is None else fw, np.nan if se is None else se)

_WORKER = {'lorentzian': _mc_lorentzian, 'pseudo_voigt': _mc_pseudo_voigt, 'gregor': _mc_gregor}

def _init_worker(): torch.set_num_threads(1)

def cloud_at_node(mu, gamma, est_key, n_mc, seed, pool):
    """2-D (FWHM, sigma_fit) cloud for one grid node and one estimator.

    The (u, b) noise draws depend only on (mu, sigma_prop, lambda) and the seed, so every estimator
    sees the SAME draws -> the comparison is paired.  mu enters only through the photon count n.
    """
    rng = np.random.default_rng(seed)
    tasks = []
    for _ in range(n_mc):
        u, b, n = draw_fixed_noise(mu, COND_SIGMA_PROP, COND_LAMBDA, rng)
        tasks.append((gamma, u.numpy(), b.numpy()))
    res = list(pool.map(_WORKER[est_key], tasks, chunksize=8))
    f = np.array([r[0] for r in res], dtype=float)
    s = np.array([r[1] for r in res], dtype=float)
    return f, s

print('estimators + worker ready')


estimators + worker ready


In [5]:
# ============================================================
# PRECOMPUTE — the whole (mu, gamma) grid for all three estimators
#   (this is the long cell; the raw clouds live in memory, the binned densities + stats are saved)
# ============================================================
CLOUDS = {}        # (est_key, i, j) -> (FWHM, sigma) arrays
STATS  = {}        # (est_key, i, j) -> dict(n_ok, f_med, f_iqr, f_std, s_med, s_iqr)

def _stats(f, s):
    ok = np.isfinite(f) & np.isfinite(s)
    fo, so = f[ok], s[ok]
    if fo.size == 0:
        return dict(n_ok=0, f_med=np.nan, f_iqr=np.nan, f_std=np.nan, s_med=np.nan, s_iqr=np.nan)
    q = np.percentile
    return dict(n_ok=int(fo.size),
                f_med=float(np.median(fo)), f_iqr=float(q(fo, 75) - q(fo, 25)), f_std=float(np.std(fo)),
                s_med=float(np.median(so)), s_iqr=float(q(so, 75) - q(so, 25)))

t0 = time.time()
with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=_init_worker) as pool:
    for est_name, est_key in ESTIMATORS:
        te = time.time()
        for i, mu in enumerate(GRID_MU):
            for j, gamma in enumerate(GRID_GAMMA):
                f, s = cloud_at_node(mu, gamma, est_key, N_MC, SEED + 1000 * i + j, pool)
                CLOUDS[(est_key, i, j)] = (f, s)
                STATS[(est_key, i, j)] = _stats(f, s)
        print(f'  {est_name:<28} done in {(time.time()-te)/60:.1f} min', flush=True)

print(f'\nprecompute total: {(time.time()-t0)/60:.1f} min  |  nodes: {len(CLOUDS)}')

# ---- save companion run data (binned densities + per-node stats) for re-rendering ----
import numpy as _np
if SMOKE:
    print('smoke run: skipping companion-data save')
else:
    os.makedirs('data/processed', exist_ok=True)
    _payload = {'mu_grid': _np.asarray(GRID_MU, dtype=float),
                'gamma_grid': _np.asarray(GRID_GAMMA, dtype=float),
                'estimators': _np.asarray([k for _, k in ESTIMATORS])}
    _stats_mat = _np.full((len(ESTIMATORS), len(GRID_MU), len(GRID_GAMMA), 6), _np.nan)
    for ei, (_, est_key) in enumerate(ESTIMATORS):
        for i in range(len(GRID_MU)):
            for j in range(len(GRID_GAMMA)):
                f, s = CLOUDS[(est_key, i, j)]
                ok = _np.isfinite(f) & _np.isfinite(s)
                H, _, _ = _np.histogram2d(f[ok], s[ok], bins=[40, 40], range=[[0, 70], [0, 40]])
                _payload[f'dens_{ei}_{i}_{j}'] = H.astype(_np.uint16)
                d = STATS[(est_key, i, j)]
                _stats_mat[ei, i, j] = [d['n_ok'], d['f_med'], d['f_iqr'], d['f_std'], d['s_med'], d['s_iqr']]
    _payload['stats'] = _stats_mat
    try:
        _np.savez_compressed('data/processed/22e_clouds.npz', **_payload)
        print('saved data/processed/22e_clouds.npz')
    except Exception as _e:
        print('save skipped:', _e)


  ours: Lorentzian MLE         done in 77.2 min


  ours: pseudo-Voigt fit       done in 299.4 min


  Gregor: binned Voigt LSQ     done in 41.3 min



precompute total: 417.9 min  |  nodes: 390


saved data/processed/22e_clouds.npz


In [6]:
# ============================================================
# FIG 1 — grid heatmaps per estimator: median FWHM, median sigma_fit, IQR(FWHM)
# ============================================================
def _grid_matrix(est_key, key):
    Z = np.full((len(GRID_MU), len(GRID_GAMMA)), np.nan)
    for i in range(len(GRID_MU)):
        for j in range(len(GRID_GAMMA)):
            Z[i, j] = STATS[(est_key, i, j)][key]
    return Z

fig = make_subplots(rows=len(ESTIMATORS), cols=3,
                    subplot_titles=[f'{est_name} — {stat}' for est_name, _ in ESTIMATORS
                                    for stat in ('median FWHM (MHz)', 'median sigma_fit (MHz)', 'IQR FWHM (MHz)')],
                    horizontal_spacing=0.06, vertical_spacing=0.10)
for r, (est_name, est_key) in enumerate(ESTIMATORS, start=1):
    for c, key in enumerate(('f_med', 's_med', 'f_iqr'), start=1):
        fig.add_trace(go.Heatmap(x=[f'{g:g}' for g in GRID_GAMMA], y=[f'{m:g}' for m in GRID_MU],
                                 z=_grid_matrix(est_key, key), colorscale='Viridis',
                                 showscale=(c == 3), colorbar=dict(title='MHz') if c == 3 else None,
                                 hovertemplate='mu=%{y}<br>gamma=%{x}<br>%{z:.2f}<extra></extra>'), r, c)
    fig.update_xaxes(title_text='gamma (MHz)', row=r, col=1)
    fig.update_yaxes(title_text='mu (photons)', row=r, col=1)
fig.update_layout(height=260 * len(ESTIMATORS), width=1050, title_text='FIG 1 — the (mu, gamma) map', margin=dict(t=90, b=40))
fig.show()


In [7]:
# ============================================================
# FIG 2 — small-multiples: the 2-D (FWHM, sigma_fit) cloud at a coarse (mu, gamma) sub-grid
#   (our current estimator = Lorentzian MLE; the real 1nW Trans60 data overlaid in red for reference)
# ============================================================
def _real_data(exp):
    d = np.genfromtxt(exp['data_file'])
    f = d[:, 0] * 1000.0; e = d[:, 1] * 1000.0
    ok = ~np.isnan(f) & ~np.isnan(e) & (f > 0)
    flt = ok & (e / f < 10.0)
    return f[flt], e[flt]

SUB_MU    = [GRID_MU[i] for i in SUB_IDX_MU]
SUB_GAMMA = [GRID_GAMMA[j] for j in SUB_IDX_GAMMA]
def _grid_range(v, pct=(1, 99), pad=0.15):
    lo, hi = np.percentile(v, pct); d = max(hi - lo, 1e-6)
    return lo - pad * d, hi + pad * d
ref = next(e for e in EXPERIMENTS if e['name'] == '1nW Trans60')
rf, rs = _real_data(ref)
X_RANGE = (0.0, 70.0); Y_RANGE = (0.0, 40.0)

def _dens_xy(f, s, xr=X_RANGE, yr=Y_RANGE, nx=40, ny=40):
    ok = np.isfinite(f) & np.isfinite(s) & (f >= xr[0]) & (f <= xr[1]) & (s >= yr[0]) & (s <= yr[1])
    H, xe, ye = np.histogram2d(f[ok], s[ok], bins=[nx, ny], range=[xr, yr])
    Z = gaussian_filter(H.T, 1.1); Z = Z / max(Z.max(), 1e-12)
    return 0.5 * (xe[:-1] + xe[1:]), 0.5 * (ye[:-1] + ye[1:]), Z

fig2 = make_subplots(rows=len(SUB_IDX_MU), cols=len(SUB_IDX_GAMMA), horizontal_spacing=0.04, vertical_spacing=0.05,
                     subplot_titles=[f'mu={GRID_MU[i]:g}, gamma={GRID_GAMMA[j]:g}'
                                     for i in SUB_IDX_MU for j in SUB_IDX_GAMMA])
for r, i in enumerate(SUB_IDX_MU, start=1):
    for c, j in enumerate(SUB_IDX_GAMMA, start=1):
        f, s = CLOUDS[('lorentzian', i, j)]
        xc, yc, Z = _dens_xy(f, s)
        fig2.add_trace(go.Heatmap(x=xc, y=yc, z=Z, colorscale='Blues', showscale=False), r, c)
        fig2.add_trace(go.Scatter(x=rf, y=rs, mode='markers',
                                  marker=dict(size=2, color='#e63946', opacity=0.35), showlegend=False), r, c)
        fig2.update_xaxes(range=X_RANGE, row=r, col=c); fig2.update_yaxes(range=Y_RANGE, row=r, col=c)
        if c == 1: fig2.update_yaxes(title_text='sigma_fit (MHz)', row=r, col=c)
        if r == len(SUB_IDX_MU): fig2.update_xaxes(title_text='FWHM (MHz)', row=r, col=c)
fig2.update_layout(height=250 * len(SUB_IDX_MU), width=max(760, 200 * len(SUB_IDX_GAMMA)),
                   title_text='FIG 2 — cloud vs (mu, gamma), our Lorentzian MLE (red = real 1nW T60)', margin=dict(t=80, b=40))
fig2.show()


In [8]:
# ============================================================
# FIG 3 — estimator overlay at representative nodes (paired draws)
# ============================================================
NODES_DISPLAY = NODES
COLORS = {'lorentzian': '#1d3557', 'pseudo_voigt': '#2a9d8f', 'gregor': '#e76f51'}
fig3 = make_subplots(rows=1, cols=len(NODES), horizontal_spacing=0.06,
                     subplot_titles=[f'mu={GRID_MU[i]:g}, gamma={GRID_GAMMA[j]:g}' for i, j in NODES])
for c, (i, j) in enumerate(NODES, start=1):
    for est_name, est_key in ESTIMATORS:
        f, s = CLOUDS[(est_key, i, j)]
        xc, yc, Z = _dens_xy(f, s)
        fig3.add_trace(go.Contour(x=xc, y=yc, z=Z, showscale=False, contours=dict(showlines=True),
                                  line=dict(width=0.6, color=COLORS[est_key]), colorscale=[[0, 'rgba(0,0,0,0)'], [1, COLORS[est_key]]],
                                  opacity=0.55, name=est_name, showlegend=(c == 1)), 1, c)
    fig3.add_trace(go.Scatter(x=rf, y=rs, mode='markers', marker=dict(size=3, color='#e63946', opacity=0.4),
                              name='real 1nW T60', showlegend=(c == 1)), 1, c)
    fig3.update_xaxes(title_text='FWHM (MHz)', row=1, col=c); fig3.update_yaxes(title_text='sigma_fit (MHz)', row=1, col=c)
fig3.update_layout(height=430, width=1200, title_text='FIG 3 — estimator comparison (same draws)', margin=dict(t=80, b=40))
fig3.show()
print('estimator medians at node (mu=%g, gamma=%g):' % (GRID_MU[NODES[1][0]], GRID_GAMMA[NODES[1][1]]))
i, j = NODES[1]
for est_name, est_key in ESTIMATORS:
    st = STATS[(est_key, i, j)]
    print(f"  {est_name:<28} FWHM {st['f_med']:.2f} +/- {st['f_std']:.2f} | sigma_fit {st['s_med']:.2f} (n_ok {st['n_ok']}/{N_MC})")
print(f"  real 1nW T60                 FWHM {np.median(rf):.2f} +/- {np.std(rf):.2f} | fit_err {np.median(rs):.2f}")


estimator medians at node (mu=24, gamma=8.5):
  ours: Lorentzian MLE         FWHM 19.49 +/- 17.37 | sigma_fit 1.14 (n_ok 2000/2000)
  ours: pseudo-Voigt fit       FWHM 34.78 +/- 39.73 | sigma_fit 2.19 (n_ok 2000/2000)
  Gregor: binned Voigt LSQ     FWHM 8.47 +/- 6.87 | sigma_fit 3.46 (n_ok 1652/2000)
  real 1nW T60                 FWHM 16.17 +/- 5.71 | fit_err 0.86


In [9]:
# ============================================================
# FIG 4 (interactive) — the explorer
#   knobs: mu and gamma (absolute values; sliders + - / + buttons)
#   dropdowns: real data (off or one of the 14 experiments) and fitter (our two vs Gregor's)
#   NOTE: needs a live kernel (VS Code / JupyterLab) — the static FIG 1-3 above are the durable view.
# ============================================================
import ipywidgets as widgets
from ipywidgets import interactive_output
from IPython.display import display

REAL = {e['name']: _real_data(e) for e in EXPERIMENTS}

def explorer_figure(est_key, i_mu, i_gam, real_name):
    est_name = next(n for n, k in ESTIMATORS if k == est_key)
    mu, gam = GRID_MU[i_mu], GRID_GAMMA[i_gam]
    f, s = CLOUDS[(est_key, i_mu, i_gam)]
    xc, yc, Z = _dens_xy(f, s)
    fig = make_subplots(rows=1, cols=2, column_widths=[0.72, 0.28], horizontal_spacing=0.08,
                        subplot_titles=[f'(FWHM, sigma_fit) cloud — mu={mu:g} photons, gamma={gam:g} MHz  [{est_name}]',
                                        'grid position'])
    fig.add_trace(go.Heatmap(x=xc, y=yc, z=Z, colorscale='Blues', showscale=False), 1, 1)
    fig.add_trace(go.Scatter(x=f, y=s, mode='markers', marker=dict(size=3, color='rgba(69,123,157,0.25)'), name='sim draws'), 1, 1)
    if real_name != 'off':
        rx, ry = REAL[real_name]
        fig.add_trace(go.Scatter(x=rx, y=ry, mode='markers', marker=dict(size=4, color='#e63946', opacity=0.5),
                                 name=f'real {real_name}'), 1, 1)
    Zg = np.zeros((len(GRID_MU), len(GRID_GAMMA))); Zg[i_mu, i_gam] = 1.0
    fig.add_trace(go.Heatmap(x=[f'{g:g}' for g in GRID_GAMMA], y=[f'{m:g}' for m in GRID_MU], z=Zg,
                             colorscale=[[0, 'white'], [1, '#e63946']], showscale=False), 1, 2)
    fig.update_xaxes(title_text='FWHM (MHz)', range=X_RANGE, row=1, col=1)
    fig.update_yaxes(title_text='sigma_fit (MHz)', range=Y_RANGE, row=1, col=1)
    fig.update_xaxes(title_text='gamma (MHz)', row=1, col=2)
    fig.update_yaxes(title_text='mu (photons)', row=1, col=2)
    fig.update_layout(height=470, margin=dict(t=70, b=40), showlegend=True)
    return fig

mu_vals = [f'{m:g}' for m in GRID_MU]; gam_vals = [f'{g:g}' for g in GRID_GAMMA]
mu_sl = widgets.SelectionSlider(options=[(f'{m:g}', i) for i, m in enumerate(GRID_MU)], value=NODES[1][0], description='mu (photons)', continuous_update=False)
gam_sl = widgets.SelectionSlider(options=[(f'{g:g}', j) for j, g in enumerate(GRID_GAMMA)], value=NODES[1][1], description='gamma (MHz)', continuous_update=False)
est_dd = widgets.Dropdown(options=[(n, k) for n, k in ESTIMATORS], value='lorentzian', description='fitter')
real_dd = widgets.Dropdown(options=['off'] + [e['name'] for e in EXPERIMENTS], value='1nW Trans60', description='real data')

def _bump(sl, d):
    def f(_): sl.value = int(np.clip(sl.value + d, sl.options[0][1], sl.options[-1][1]))
    return f
b_mum = widgets.Button(description='mu -', layout=widgets.Layout(width='70px')); b_mup = widgets.Button(description='mu +', layout=widgets.Layout(width='70px'))
b_gam = widgets.Button(description='gam -', layout=widgets.Layout(width='70px')); b_gap = widgets.Button(description='gam +', layout=widgets.Layout(width='70px'))
b_mum.on_click(_bump(mu_sl, -1)); b_mup.on_click(_bump(mu_sl, +1)); b_gam.on_click(_bump(gam_sl, -1)); b_gap.on_click(_bump(gam_sl, +1))

out = interactive_output(lambda i_mu, i_gam, est, real: explorer_figure(est, i_mu, i_gam, real).show(),
                         {'i_mu': mu_sl, 'i_gam': gam_sl, 'est': est_dd, 'real': real_dd})
display(widgets.VBox([
    widgets.HBox([b_mum, b_mup, mu_sl]),
    widgets.HBox([b_gam, b_gap, gam_sl]),
    widgets.HBox([est_dd, real_dd]),
    out,
]))


## Notes / how to read
- **gamma** mostly moves the cloud along **FWHM ~ 2*gamma** and rescales sigma_fit.
- **mu** mostly **compresses** the cloud (more photons -> tighter FWHM and smaller sigma_fit); the width is
  set by the relative count noise `sigma_prop/mu`. This is why mu is weakly identified from FWHM alone.
- **Estimator matters**: our unbinned Lorentzian MLE sits near the CRLB (narrow FWHM spread, small sigma_fit),
  our pseudo-Voigt widens sigma_fit, and Gregor's binned Voigt LSQ reproduces the real FWHM spread but
  overshoots sigma_fit. Compare FIG 3 and the fitter dropdown against the real red cloud.
- The pinned condition (`1nW Trans60`) fixes the count-noise scale; the real-data dropdown is directly
  comparable to the cloud only for that experiment.
- Companion run data: `data/processed/22e_clouds.npz` (binned densities + per-node stats).
